In [1]:
import os
OPENAI_KEY = os.getenv("openai_key")

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openai:gpt-4o-mini",
    openai_api_key=os.getenv("openai_key"),
    temperature=0.7,
)

In [3]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

prompt = ChatPromptTemplate([
    ("system", """Act as a world class Machine Learning engineer.
Use italian language.
Ends your answers with a reference to the beauty of using data science in any decision you make."""),
    ("user", "{query}"),
])

# Parserizzazione degli Output

I parser sono degli elementi di LangChain che possiamo utilizzare per trasformare l’output grezzo di un modello in un formato strutturato e più facilmente utilizzabile nel nostro codice.

Quando un LLM restituisce una risposta, questa può essere passata attraverso un parser che può aiutarci, tra le varie cose a interpretare e validare quell’output, estrarre informazioni chiave, trasformare il testo in dizionari, oggetti JSON, liste, o classi Python, etc

Esistono diverse tipologie di Parser che possiamo utilizzare a seconda del tipo di output atteso e del livello di struttura che vogliamo ottenere

### String Output Parser

In [4]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

chain = prompt | model | output_parser

chain.invoke({"query": "Quali sono i primi 3 pianeti del Sistema Solare?"})

'I primi tre pianeti del Sistema Solare, partendo dal Sole, sono Mercurio, Venere e Terra. Mercurio è il pianeta più vicino al Sole, seguito da Venere, spesso chiamato "il pianeta della bellezza", e infine la Terra, il nostro affascinante pianeta blu. \n\nLa bellezza di esplorare l\'universo e la scienza dei pianeti risiede nella possibilità di utilizzare i dati per prendere decisioni informate, proprio come nella data science!'

### Lista di valori

In [5]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

output_parser = CommaSeparatedListOutputParser()

format_instructions = output_parser.get_format_instructions()

prompt = PromptTemplate(
    template="{query}.\n{format_instructions}",
    input_variables=["query"],
    partial_variables={"format_instructions": format_instructions},
)

In [6]:
print(format_instructions)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [7]:
prompt.invoke({"query": "elenca i pianeti del sistema solare in ordine dal più vicino al più lontano dal Sole"}).text

'elenca i pianeti del sistema solare in ordine dal più vicino al più lontano dal Sole.\nYour response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'

In [8]:
chain = prompt | model 
res = chain.invoke({"query": "elenca i pianeti del sistema solare in ordine dal più vicino al più lontano dal Sole"})

print( type(res) )
print("---")
print(res)

print("\n===\n")

print( type(res.content) )
print("---")
print(res.content)

<class 'langchain_core.messages.ai.AIMessage'>
---
content='Mercurio, Venere, Terra, Marte, Giove, Saturno, Urano, Nettuno' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 56, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_51db84afab', 'id': 'chatcmpl-Cb3VksQkzeopoI1sFwzV4D62iKrZ8', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--1dd4b30c-b537-4908-8f0a-fc2d8b450d73-0' usage_metadata={'input_tokens': 56, 'output_tokens': 21, 'total_tokens': 77, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

===

<class 'str'>
---
Mercurio, Venere, Terra, Marte, Giove, Saturno,

In [9]:
chain = prompt | model | output_parser
res = chain.invoke({"query": "elenca i pianeti del sistema solare in ordine dal più vicino al più lontano dal Sole"})

print(type (res) )
print("---")
print(res)

<class 'list'>
---
['Mercurio', 'Venere', 'Terra', 'Marte', 'Giove', 'Saturno', 'Urano', 'Nettuno']


## Output strutturato in LangChain 1.0

In molti casi, quando utilizzi un modello di linguaggio all’interno di una pipeline (ad esempio per chatbot, agent, estrazione dati, RAG, ecc.), vuoi che l’output non sia semplicemente un blocco di testo libero, ma sia **organizzato** in una forma nota e prevedibile (dictionary, JSON, oggetto Pydantic…).
Questo è ciò che si intende per *output strutturato*.
La libreria LangChain 1.0 introduce un supporto formale per tale modalità, mediante il metodo `with_structured_output()` e/o parser dedicati.

### Perché usare l’output strutturato

* Permette di definire uno **schema** (campi, tipi, descrizioni) che l’output dovrà rispettare.
* Favorisce l’integrazione downstream (es. inserimento in database, orditura di oggetti, logica condizionale) in modo programmatico, anziché dover parsare manualmente un testo libero.
* Aumenta l’affidabilità del sistema: meno “interpretazione libera” del modello, più predicibilità del formato.
* In LangChain 1.0 è possibile sfruttare modelli che supportano direttamente modalità strutturate (come JSON mode, funzione/tool-calling) grazie a `with_structured_output()`.


### Differenza rispetto al semplice “parser di output”

Il “semplice parser di output” (output parser) in LangChain pone un approccio diverso: invece di far generare al modello già un output *certamente* conforme allo schema, si prende un output di testo libero (o semi-noto) e lo si “parsifica” successivamente. Alcuni punti chiave:

* Con un output parser classico (es. `JsonOutputParser`, `StructuredOutputParser`, `PydanticOutputParser`) definisci delle istruzioni (via `get_format_instructions()`) che instradano il modello a generare un certo formato sfruttando il prompt di input, ma non hai garanzia totale che il modello rispetti lo schema.
* Questo approccio è più “flessibile” ma anche più fragile: se il modello devia dal formato richiesto (ad esempio aggiunge testo narrativo, non restituisce JSON valido…) allora il parsing può fallire.
* Al contrario, con `with_structured_output()`, quando il modello provider lo supporta, l’output viene generato *già* come struttura conforme (grazie a tool calling, JSON mode o schema nativo) e quindi il downstream è più solido.

### Tabella comparativa (concettuale)

| Approccio                     | Dove si applica                          | Punti di forza                      | Limiti                                                     |
| ----------------------------- | ---------------------------------------- | ----------------------------------- | ---------------------------------------------------------- |
| Parser di output classico     | Qualsiasi modello, testo libero          | Alta compatibilità                  | Fragile: parsing manuale, rischio errori                   |
| Output strutturato con schema | Modelli + provider che supportano schema | Maggiore affidabilità, tipizzazione | Richiede supporto del modello/provider, schema da definire |

------

Con LangChain 1.0 l’output strutturato è la modalità consigliata quando si ha bisogno di output prevedibile, di un particolare formato e facilmente integrabile con sistemi esterni.
Il semplice parser resta utile in scenari più “liberi” ma comporta maggiore rischio di deviazioni e fallimenti.

### JSON 

In [10]:
json_schema = {
    "title": "Movie",
    "description": "A movie with details",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "The title of the movie"
        },
        "year": {
            "type": "integer",
            "description": "The year the movie was released"
        },
        "director": {
            "type": "string",
            "description": "The director of the movie"
        },
        "rating": {
            "type": "number",
            "description": "The movie's rating out of 10"
        }
    },
    "required": ["title", "year", "director", "rating"]
}

model_with_structure = model.with_structured_output(json_schema, method="json_schema",)

response = model_with_structure.invoke("Provide details about the movie Inception")

print(type(response))
print(response) 

<class 'dict'>
{'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}


### TypedDict

In [11]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_structure = model.with_structured_output(MovieDict)

response = model_with_structure.invoke("Provide details about the movie Inception")

print(type(response))
print(response) 

<class 'dict'>
{'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}


## Pydantic

In [12]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie)

response = model_with_structure.invoke("Provide details about the movie Inception")

print(type(response))
print(response)

<class '__main__.Movie'>
title='Inception' year=2010 director='Christopher Nolan' rating=8.8


In [13]:
model_with_structure.invoke(" film 'La vita è bella' del 1997  diretto da Roberto Benigni con voto 8.6 ")

Movie(title='La vita è bella', year=1997, director='Roberto Benigni', rating=8.6)

In [14]:
model_with_structure.invoke(" 'La vita è bella' , 1997, Roberto Benigni , otto ")

Movie(title='La vita è bella', year=1997, director='Roberto Benigni', rating=8.0)